# Regularization in Linear Regression

Goal of this notebook:
- Build a regression model on a synthetic **car price** dataset with:
  - features on very different scales
  - some linear, some non-linear relationships
  - some completely random (irrelevant) features
- See **overfitting** with high-degree polynomial features.
- Use **Ridge regularization** (L2) with cross-validation to control overfitting.
- Show why **feature scaling** is crucial when using regularization.


In [9]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, r2_score

import matplotlib.pyplot as plt

np.random.seed(42)

ImportError: cannot import name 'root_mean_squared_error' from 'sklearn.metrics' (c:\Users\petra\anaconda3\envs\ml\lib\site-packages\sklearn\metrics\__init__.py)

## 1. Create a synthetic car price dataset

We simulate a **car price** dataset with:
- features on different scales
- a mix of linear and non-linear relations to the target
- a few purely random (noise) features


In [ ]:
n_samples = 1000
rng = np.random.default_rng(42)

# Features with very different scales
age_years = rng.uniform(0, 15, size=n_samples)             # 0–15 years
mileage_kkm = rng.uniform(5, 250, size=n_samples)    # 5000–25000 km 
engine_size = rng.uniform(1.0, 4.5, size=n_samples)        # liters
brand_premium_index = rng.integers(0, 4, size=n_samples)   # 0–3 (budget → premium)
horsepower = rng.normal(120, 40, size=n_samples)
horsepower = np.clip(horsepower, 60, 300)
safety_rating = rng.integers(1, 6, size=n_samples)         # 1–5


# Non-linear-ish ground truth for price
base_price = 15000
price = (
    base_price
    + 1800 * engine_size
    - 900 * age_years
    - 0.85 * mileage_kkm
    + 4500 * brand_premium_index
    + 55 * horsepower
    + 1200 * safety_rating
    + 3000 * (engine_size ** 2)      # non-linear effect
    - 120 * (age_years ** 2)         # non-linear effect
)

# Add noise to price
price = price + rng.normal(0, 5000, size=n_samples)

X = pd.DataFrame({
    'age_years': age_years,
    'mileage_kkm': mileage_kkm,
    'engine_size': engine_size,
    'brand_premium_index': brand_premium_index,
    'horsepower': horsepower,
    'safety_rating': safety_rating,
    'noise1': rng.normal(0, 1, size=n_samples),  # additional noise feature,
    'noise2': rng.normal(0, 1, size=n_samples),  # additional noise feature,
    'noise3': rng.normal(0, 1, size=n_samples),  # additional noise feature,
    'noise4': rng.normal(0, 1, size=n_samples),  # additional noise feature
})
y = price

X.head()

In [ ]:
# price histogram

plt.figure(figsize=(8, 3))
plt.hist(y, bins=30)
plt.xlabel('Price') 
plt.ylabel('Frequency')
plt.title('Histogram of Car Prices')    


## 2. Train–test split

We keep a **test set** for final evaluation and only use the **training set** for fitting and cross-validation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)
X_train.shape, X_test.shape


In [ ]:
results = []
def evaluate_model(name, model, X_train, X_test, y_train, y_test, results):
    """Fits the model and appends evaluation metrics to results list."""
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    results.append({    # Append results. Note: results is modified in place
        'model': name,
        'train_R2': r2_score(y_train, y_pred_train),
        'test_R2': r2_score(y_test, y_pred_test),
        'train_RMSE': root_mean_squared_errorr(y_train, y_pred_train, squared=False),
        'test_RMSE': root_mean_squared_error(y_test, y_pred_test, squared=False),
    })

## 3. Baseline: Plain linear regression

This model uses the original features only (no polynomial expansion, no regularization).

In [ ]:
# Baseline: Linear Regression without polynomial features or regularization
baseline_lr = LinearRegression()
evaluate_model(
    'Linear (no poly, no reg)',
    baseline_lr,
    X_train, X_test, y_train, y_test,
    results,
)
pd.DataFrame(results)

## 4. Polynomial regression (overfitting expected)

We now generate **high-degree polynomial features**. With many features and some noise variables,
plain linear regression is expected to **overfit**.


In [ ]:
degree = 4  # the polynomial degree
#  for degree=4 and number of features = 10, 1000 polynomial features are generated (including all interactions up to 4th degree)

poly_lr = Pipeline([
    ('poly', PolynomialFeatures(degree=degree, include_bias=False)),   # polynomial features are generated and added to the dataset
    ('linreg', LinearRegression()),
])

evaluate_model(
    f'Poly d={degree} + Linear (no reg)',
    poly_lr,
    X_train, X_test, y_train, y_test,
    results,
)
pd.DataFrame(results)

## 5. Polynomial features + Ridge regularization (no scaling)

We now add **Ridge** (L2) regularization and tune the regularization strength $\lambda$ (alpha)
with **cross-validation**.

⚠️ Here we **do not scale** the features first, so Ridge will "see" features on very different scales.

In [ ]:
alphas = np.logspace(-4, 4, 20)

poly_ridge = Pipeline([
    ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
    ('ridge', Ridge()),
])

param_grid = {'ridge__alpha': alphas}            # hyperparameter grid
grid_ridge = GridSearchCV(
    poly_ridge,                                  # the pipeline
    param_grid,                                  # the hyperparameter grid
    cv=5,                                        # 5-fold cross-validation                  
    scoring='neg_mean_squared_error',            # we want to minimize MSE
    n_jobs=-1,                                   # utilize all CPU cores
)

grid_ridge.fit(X_train, y_train)
best_ridge = grid_ridge.best_estimator_

print('Best alpha (no scaling):', grid_ridge.best_params_['ridge__alpha'])

evaluate_model(
    f'Poly d={degree} + Ridge (no scaling)',
    best_ridge,
    X_train, X_test, y_train, y_test,
    results,
)
pd.DataFrame(results)

## 6. Polynomial features + scaling + Ridge (with CV)

Now we add a **StandardScaler** before Ridge, so that all (polynomial) features have
comparable scale. This is the setting where Ridge behaves as intended.

This model is expected to give the **best generalization performance**.

In [ ]:
scaled_poly_ridge = Pipeline([
    ('poly', PolynomialFeatures(degree=degree, include_bias=False)),   # polynomial features
    ('scaler', StandardScaler()),                                      # feature scaling
    ('ridge', Ridge()),                                                # ridge regression
])

param_grid_scaled = {'ridge__alpha': alphas}
grid_ridge_scaled = GridSearchCV(
    scaled_poly_ridge,
    param_grid_scaled,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)

grid_ridge_scaled.fit(X_train, y_train)
best_ridge_scaled = grid_ridge_scaled.best_estimator_                 # the best model found by grid search

print('Best alpha (with scaling):', grid_ridge_scaled.best_params_['ridge__alpha'])

evaluate_model(
    f'Poly d={degree} + Scaling + Ridge',
    best_ridge_scaled,
    X_train, X_test, y_train, y_test,
    results,
)

df_results = pd.DataFrame(results)


In [ ]:
# This takes a while to run..., you can skip this part if needed
# ElasticNet (with scaling)
scaled_poly_elastic_net = Pipeline([
    ('poly', PolynomialFeatures(degree=degree, include_bias=False)),   # polynomial features
    ('scaler', StandardScaler()),                                      # feature scaling
    ('elasticnet', ElasticNet(max_iter=10000, random_state=0)),        # ElasticNet estimator
])

# tune both alpha and l1_ratio for ElasticNet
param_grid_scaled = {
    'elasticnet__alpha': alphas,
    'elasticnet__l1_ratio': [0.1, 0.5, 0.9],
}

grid_elastic_scaled = GridSearchCV(
    scaled_poly_elastic_net,
    param_grid_scaled,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)

grid_elastic_scaled.fit(X_train, y_train)
best_elastic_scaled = grid_elastic_scaled.best_estimator_                 # the best model found by grid search

print('Best params (with scaling, ElasticNet):', grid_elastic_scaled.best_params_)

evaluate_model(
    f'Poly d={degree} + Scaling + ElasticNet',
    best_elastic_scaled,
    X_train, X_test, y_train, y_test,
    results,
)

df_results = pd.DataFrame(results)




In [ ]:
df_results
#df_results.sort_values('test_RMSE')

| Model                          | What it shows                                                                                                              | Takeaway                                                                                                  |
|--------------------------------|----------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------|
| Linear (no poly, no reg)       | Baseline performs decently. Bias is present (limited flexibility), but generalizes quite well.                            | Simplicity can generalize better than complex but unstable models.                                       |
| Poly d=4 + Linear (no reg)     | Train R² ≈ 1.0, but test R² < 0 — pure overfitting. The model memorized the training set and fails on new data.          | Unregularized polynomial regression explodes in variance.                                                |
| Poly d=4 + Ridge (no scaling)  | Regularization helps but can’t rescue performance since feature scales distort penalty strength.                          | Regularization without normalization is misleading — coefficients on large-scale features get unfairly penalized. |
| Poly d=4 + Scaling + Ridge     | Balanced bias–variance tradeoff: train ≈ 0.96 R², test ≈ 0.94 R².                                                         | Proper scaling makes the regularization meaningful and stabilizes learning. Best generalization.         |
| Poly d=4 + Scaling + ElasticNet| Similar performance to Ridge, showing that L1 (Lasso) component adds little value here. | In this scenario, L2 regularization suffices; L1 does not significantly enhance performance.            |

## 7. Discussion prompts

Things to inspect and discuss in class:

1. **Overfitting**:
   - Compare `train_R2` vs `test_R2` for
     - `Linear (no poly, no reg)`
     - `Poly d=... + Linear (no reg)`.
   - Where do you see clear overfitting?

2. **Effect of Ridge without scaling**:
   - Does Ridge help compared to the unregularized polynomial model?
   - Check how much it improves test error.
   - Why might it still be suboptimal when features are on different scales?

3. **Effect of scaling**:
   - Compare `Poly d=... + Ridge (no scaling)` vs `Poly d=... + Scaling + Ridge`.
   - Which one generalizes best (lowest test RMSE / highest test R²)?
   - Connect this to the definition of the Ridge penalty
     $\lambda \sum_j w_j^2$ and the role of feature scales.

4. **Playground suggestions**:
   - Change the polynomial degree.
   - Add an additional noise feature.
   - Try Lasso instead of Ridge.
   - Visualize the distribution of coefficients as you change $\lambda$.
